In [1]:
import tiktoken

In [2]:
with open("harrypotter.txt", "r", encoding="utf-8") as harryPotterBook:
    bookContent = harryPotterBook.read()

tokenizer = tiktoken.get_encoding("gpt2")
encoded_text = tokenizer.encode(bookContent)
len(encoded_text)

113793

In [4]:
encoded_sample = encoded_text[100:]
encoded_text[:10]

[18308, 14179, 290, 262, 30467, 338, 8026, 220, 198, 198]

# Simple example to create input-output pairs

In [6]:
context_length = 4 # The context length determines how many tokens to include in the input at a time

x = encoded_sample[:context_length]
y = encoded_sample[1:context_length+1]

print(f"x: {x}")
print(f"y: {y}")

x: [20935, 654, 11, 543]
y: [654, 11, 543, 925]


In [8]:
for i in range(context_length):
    x = encoded_sample[:i+1]
    y = encoded_sample[i+1]

    print(f"{x} ------> {y}")

[20935] ------> 654
[20935, 654] ------> 11
[20935, 654, 11] ------> 543
[20935, 654, 11, 543] ------> 925


# Implementing efficient data loader fetching using the PyTorch's DataLoader and Dataset classes

In [10]:
import torch
from torch.utils.data import DataLoader, Dataset

class GPTDatasetV1(Dataset):
    def __init__(self, text, tokenizer, context_length, stride):
        self.input_ids = []
        self.target_ids = []

        # Tokenization
        encoded_text = tokenizer.encode(text, allowed_special={"<|endoftext|>"})

        # Using the sliding window approach to chunk the book content using squences of context_length
        for i in range(0, len(encoded_text)-context_length, stride):
            input_chunk = encoded_text[i:i+context_length]
            target_chunk = encoded_text[i+1:i+context_length+1]
            self.input_ids.append(torch.tensor(input_chunk))
            self.target_ids.append(torch.tensor(target_chunk))

    def __len__(self):
        return len(self.input_ids)

    def __getitem__(self, index):
        return self.input_ids[index], self.target_ids[index]

In [11]:
def create_dataloader_v1(text, batch_size=4, context_length=256, stride=128, shuffle=True, drop_last=True, num_workers=0):
    # Initialize the tokenizer
    tokenizer = tiktoken.get_encoding("gpt2")

    # Create dataset using the GPTDatasetV1
    dataset = GPTDatasetV1(text, tokenizer, context_length, stride)

    # Create dataloader using the DataLoader from pytorch
    dataloader = DataLoader(
        dataset=dataset,
        batch_size=batch_size,
        shuffle=shuffle,
        drop_last=drop_last,
        num_workers=num_workers
    )

    return dataloader

# Testing the DataLoader with a batch_size of 1 for an LLM with context_length of 4

In [12]:
with open("harrypotter.txt", "r", encoding="utf-8") as harry_potter_book:
    raw_text = harry_potter_book.read()

In [13]:
dataloader_1 = create_dataloader_v1(raw_text, batch_size=1, context_length=4, stride=1, shuffle=False)

data_iterator = iter(dataloader_1)
first_batch = next(data_iterator)
first_batch

[tensor([[18308, 14179,   290,   262]]),
 tensor([[14179,   290,   262, 30467]])]

In [14]:
second_batch = next(data_iterator)
second_batch

[tensor([[14179,   290,   262, 30467]]),
 tensor([[  290,   262, 30467,   338]])]

# Testing the DataLoader with a batch_size of 8 for an LLM with context_length of 4

In [15]:
dataloader_2 = create_dataloader_v1(raw_text, batch_size=8, context_length=4, stride=1, shuffle=False)

data_iterator = iter(dataloader_2)
first_batch = next(data_iterator)
first_batch

[tensor([[18308, 14179,   290,   262],
         [14179,   290,   262, 30467],
         [  290,   262, 30467,   338],
         [  262, 30467,   338,  8026],
         [30467,   338,  8026,   220],
         [  338,  8026,   220,   198],
         [ 8026,   220,   198,   198],
         [  220,   198,   198, 41481]]),
 tensor([[14179,   290,   262, 30467],
         [  290,   262, 30467,   338],
         [  262, 30467,   338,  8026],
         [30467,   338,  8026,   220],
         [  338,  8026,   220,   198],
         [ 8026,   220,   198,   198],
         [  220,   198,   198, 41481],
         [  198,   198, 41481, 16329]])]

***Note:** The input size (context_length) of 4 is relatively very small and only chosen for demonstration purpose in the code. It is common the train LLMs with input size (context_length) of minimum 256* 

<div class="alert-warning">
The small batch size require less memory during training but leads to noisy weight updates.<br/>
Just like in regular deep learning, the batch size is a trade-off and a hyperparameter to experiment with when training LLM.
</div>

# Testing the DataLoader with a batch_size of 8 for an LLM with context_length of 256 (common in training LLMs) and a stride of 128

In [16]:
dataloader_3 = create_dataloader_v1(raw_text, batch_size=4, context_length=256, stride=128, shuffle=False)

data_iterator = iter(dataloader_3)
first_batch = next(data_iterator)
first_batch

[tensor([[18308, 14179,   290,  ...,   484,  8020,   470],
         [   13,  9074,    13,  ...,  9074,    13,   360],
         [ 1138,   329,  1811,  ...,   319,   262, 19353],
         [ 1834,  1636, 19092,  ...,  5228,   286,  4389]]),
 tensor([[14179,   290,   262,  ...,  8020,   470,  1138],
         [ 9074,    13,   360,  ...,    13,   360,  1834],
         [  329,  1811,   812,  ...,   262, 19353,    11],
         [ 1636, 19092,   510,  ...,   286,  4389, 16809]])]

In [17]:
second_batch = next(data_iterator)
second_batch

[tensor([[   11,   290,  3088,  ...,   257,  1588,  1502],
         [16809,  9974,    11,  ..., 48508,  6568,   306],
         [  286, 26721,   339,  ...,   736,   284,   262],
         [ 1978,    13,  1770,  ...,   618,   339,  1807]]),
 tensor([[  290,  3088,   284,  ...,  1588,  1502,   286],
         [ 9974,    11,   475,  ...,  6568,   306,  1978],
         [26721,   339,   373,  ...,   284,   262,  4324],
         [   13,  1770,    13,  ...,   339,  1807,   339]])]